# bonesim - Colab GPU benchmark

Decides one question: **is a hosted NVIDIA GPU faster than the local laptop for
this project's gprMax workload?**

gprMax ships a CUDA solver behind a `-gpu` flag. The development laptop has an
AMD RX 6800M, so that solver is unreachable there and every local figure is
CPU-only. A Colab GPU can run it.

One trade to keep in view: Colab's free tier gives about **2 vCPUs** against the
laptop's **16 threads**, so a CPU-only run here will be *slower* than the
laptop. The GPU is the entire reason to be here.

Local numbers to beat, all measured:

| workload | local |
|---|---|
| benchmark model, no dispersion | 178 M cell-updates/s |
| production model, 6 Debye tissues | 74 M cell-updates/s |
| one 61-position B-scan | 610 s |
| full matrix (192 B-scans) | 32.5 h |

Run the cells in order; about 15 minutes, most of it building gprMax.

**Set a GPU runtime first:** Runtime > Change runtime type > T4 GPU.


## 1. Confirm a GPU is attached

In [ ]:
import os
import subprocess

out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if out.returncode != 0:
    raise SystemExit(
        "No GPU attached. Runtime > Change runtime type > T4 GPU, then rerun.")
print(out.stdout)
print("CPU cores available:", os.cpu_count())


## 2. Build gprMax

From source, because the Cython extensions must match this runtime's Python and NumPy.

In [ ]:
%%bash
set -e
pip install -q "cython>=3.0" numpy scipy h5py matplotlib psutil colorama \
    terminaltables tqdm 2>&1 | tail -2
if [ ! -d /content/gprMax ]; then
  git clone -q https://github.com/gprMax/gprMax.git /content/gprMax
fi
cd /content/gprMax
python setup.py build 2>&1 | tail -3
pip install -e . --no-build-isolation -q 2>&1 | tail -2
python -c "import gprMax; print('gprMax', gprMax.__version__)"


## 3. Install pycuda

This is what enables the `-gpu` flag. If it fails, the GPU solver is unavailable and the answer is no.

In [ ]:
!pip install -q pycuda 2>&1 | tail -3

import pycuda.driver as drv

drv.init()
dev = drv.Device(0)
print("GPU:", dev.name())
print("compute capability: %d.%d" % dev.compute_capability())
print("memory: %.1f GB" % (dev.total_memory() / 1e9))


## 4. Write the benchmark models

Identical to those measured locally, so the comparison is like for like.

- **A** non-dispersive, the original throughput benchmark
- **B** the production model with six two-pole Debye tissues, which is what the matrix actually runs. Dispersion costs 2.4x locally, and the CUDA kernels do handle multi-pole dispersive materials, so B is the number that decides this.

In [ ]:
from pathlib import Path

W = Path("/content/work")
W.mkdir(exist_ok=True)

MODEL_A = '#title: non-dispersive throughput benchmark\n#domain: 0.140 0.100 0.0002\n#dx_dy_dz: 0.0002 0.0002 0.0002\n#time_window: 3e-9\n\n#material: 5.0 0.6 1 0 fatlike\n#material: 42 6.0 1 0 musclelike\n#material: 12 0.4 1 0 bonelike\n\n#box: 0 0.030 0 0.140 0.070 0.0002 musclelike\n#box: 0 0.070 0 0.140 0.074 0.0002 fatlike\n#cylinder: 0.070 0.050 0 0.070 0.050 0.0002 0.010 bonelike\n\n#waveform: gaussiandot 1 7e9 pulse\n#hertzian_dipole: z 0.070 0.086 0 pulse\n#rx: 0.070 0.086 0\n'

MODEL_B = '#title: production model, dispersive tissues\n#domain: 0.1000 0.0550 0.0002\n#dx_dy_dz: 0.0002 0.0002 0.0002\n#time_window: 2.5e-09\n\n#material: 4.57196 0.713624 1 0 skin_dry\n#add_dispersion_debye: 2 32.3758 7.43609e-12 3.66933 7.9816e-11 skin_dry\n#material: 3.01149 0.0476365 1 0 fat\n#add_dispersion_debye: 2 1.85023 8.1162e-12 0.516319 2.89757e-11 fat\n#material: 9.20366 0.867599 1 0 muscle\n#add_dispersion_debye: 2 39.6209 7.56898e-12 5.05775 2.79524e-11 muscle\n#material: 3.69273 0.118373 1 0 cortical_bone\n#add_dispersion_debye: 2 5.62145 1.04285e-11 2.75291 3.42853e-11 cortical_bone\n#material: 3.0272 0.0374966 1 0 marrow\n#add_dispersion_debye: 2 1.86144 8.27295e-12 0.510127 3.00951e-11 marrow\n#material: 9.14071 1.43319 1 0 blood\n#add_dispersion_debye: 2 43.9658 8.41017e-12 6.82598 2.90441e-11 blood\n\n#box: 0.0000 0.0350 0 0.1000 0.0370 0.0002 skin_dry\n#box: 0.0000 0.0320 0 0.1000 0.0350 0.0002 fat\n#box: 0.0000 0.0310 0 0.1000 0.0320 0.0002 muscle\n#box: 0.0000 0.0270 0 0.1000 0.0310 0.0002 cortical_bone\n#box: 0.0000 0.0170 0 0.1000 0.0270 0.0002 marrow\n#box: 0.0000 0.0130 0 0.1000 0.0170 0.0002 cortical_bone\n#box: 0.0495 0.0270 0 0.0505 0.0310 0.0002 blood\n\n#waveform: gaussiandot 1 7e9 pulse\n#hertzian_dipole: z 0.0200 0.0490 0 pulse\n#rx: 0.0200 0.0490 0\n'

(W / "model_a.in").write_text(MODEL_A)
(W / "model_b.in").write_text(MODEL_B)

# Cell counts and iteration counts, so throughput is comparable.
CELLS = {"model_a": 700 * 500, "model_b": 500 * 275}
ITERS = {"model_a": 6361, "model_b": 5299}
for k in CELLS:
    print("%s: %d cells x %d iterations = %.2e cell-updates"
          % (k, CELLS[k], ITERS[k], CELLS[k] * ITERS[k]))


## 5. Run each model on CPU and GPU

Expect poor CPU numbers - two vCPUs. They are recorded only to show the GPU speedup is real rather than an artefact of the runtime.

In [ ]:
import re
import subprocess
import time


def run(model, gpu):
    """Return (solver_seconds, None) or (None, error_tail)."""
    cmd = ["python", "-m", "gprMax", model + ".in"] + (["-gpu"] if gpu else [])
    t0 = time.time()
    p = subprocess.run(cmd, cwd="/content/work", capture_output=True, text=True)
    dt = time.time() - t0
    if p.returncode != 0:
        tail = (p.stdout + p.stderr).strip().splitlines()[-8:]
        return None, "\n".join(tail)
    # Prefer the solver time gprMax reports over wall clock, which includes
    # model build and, on the GPU, kernel compilation.
    m = re.search(r"Solving time \[HH:MM:SS\]: (\d+):(\d+):([\d.]+)", p.stdout)
    if m:
        h, mm, s = m.groups()
        dt = int(h) * 3600 + int(mm) * 60 + float(s)
    return dt, None


results = {}
for model in ("model_a", "model_b"):
    for gpu in (False, True):
        tag = model + (" GPU" if gpu else " CPU")
        dt, err = run(model, gpu)
        if err is not None:
            print("%-16s FAILED\n%s\n" % (tag, err))
            results[(model, gpu)] = None
        else:
            rate = CELLS[model] * ITERS[model] / dt
            results[(model, gpu)] = (dt, rate)
            print("%-16s %8.1f s   %7.1f M cell-updates/s"
                  % (tag, dt, rate / 1e6))


## 6. Verdict

In [ ]:
LOCAL = {"model_a": 178000000.0, "model_b": 74000000.0}
LOCAL_BSCAN_S = 610.0
MATRIX = 192

print("%-10s%12s%12s%12s%14s"
      % ("model", "local", "colab CPU", "colab GPU", "GPU vs local"))
gain = None
for model in ("model_a", "model_b"):
    cpu = results.get((model, False))
    gpu = results.get((model, True))
    lo = LOCAL[model]
    row = "%-10s%11.0fM" % (model, lo / 1e6)
    row += "%11.0fM" % (cpu[1] / 1e6) if cpu else "%12s" % "failed"
    row += "%11.0fM" % (gpu[1] / 1e6) if gpu else "%12s" % "failed"
    row += "%13.2fx" % (gpu[1] / lo) if gpu else "%14s" % "-"
    print(row)
    if model == "model_b" and gpu:
        gain = gpu[1] / lo

print()
if gain is None:
    print("The GPU solver did not run on the production model.")
    print("Keep the matrix on the laptop.")
elif gain > 1.5:
    local_h = MATRIX * LOCAL_BSCAN_S / 3600
    print("GPU is %.1fx the laptop on the production model." % gain)
    print("Full matrix: %.1f h locally -> %.1f h here." % (local_h, local_h / gain))
    print("Worth moving, but in chunks checkpointed to Drive: free sessions")
    print("cap near 12 h and idle out after about 90 minutes.")
else:
    print("GPU is only %.2fx the laptop. Not worth the session-management" % gain)
    print("cost - keep the matrix local.")


## Notes

**Session limits.** Free Colab caps a session near 12 hours and disconnects
after roughly 90 minutes idle. The matrix cannot run in one sitting whatever the
speedup, so it has to be split into chunks that write results to Drive as they
finish, with the driver skipping work already done.

**Rebuild cost.** gprMax is rebuilt from source every session. Caching the built
tree to Drive avoids paying that each time.

**What does not move.** Only the FDTD solve. Geometry, material fitting,
processing, and statistics all stay local, where iteration is fast and there are
no session limits.
